In [ ]:
import numpy as np
import pandas as pd
import scipy.io
import math
import os
import ntpath
import sys
import logging
import time
import sys

from importlib import reload
import plotly.graph_objects as go

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

from keras.models import Sequential
from keras.layers import Dense, Dropout, Activation
from keras.optimizers import SGD, Adam
#from keras.utils import np_utils
from keras.layers import LSTM, Embedding, RepeatVector, TimeDistributed, Masking, GRU
from keras.callbacks import EarlyStopping, ModelCheckpoint, LambdaCallback

In [ ]:
import os
import matplotlib.pyplot as plt
import numpy as np
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error
import tensorflow as tf

# Function to load a model and make predictions
def load_and_predict(model_path, X_test):
        
    model = tf.keras.models.load_model(model_path)
    predictions = model.predict(X_test)
    return predictions


# Function to calculate and print metrics
def calculate_and_print_metrics(y_test, predictions):
    mse = mean_squared_error(y_test, predictions)
    r2 = r2_score(y_test, predictions)
    rmse = np.sqrt(mse)
    mae = mean_absolute_error(y_test, predictions)
    

    print("Mean Squared Error (MSE):", mse)
    print("R-squared (R2):", r2)
    print("Root Mean Squared Error (RMSE):", rmse)
    print("Mean Absolute Error (MAE):", mae)
    

# Function to plot true vs predicted values
def plot_true_vs_predicted(y_test, predictions, model_name):
    plt.figure(figsize=(10, 6))
    plt.plot(y_test, label='True Values')
    plt.plot(predictions, label='Predicted Values')
    plt.xlabel('Data Points')
    plt.ylabel('Values')
    plt.legend()
    plt.title(f'True vs Predicted Values - Model: {model_name}')
    plt.show()


In [4]:
import numpy as np
import pandas as pd
import logging
import plotly.graph_objects as go
from sklearn.preprocessing import MinMaxScaler
from datetime import datetime, timedelta

DATA_PATH = 'LG_HG2_Original_Dataset_McMasterUniversity_Jan_2020//'

class LgData():
    def __init__(self, base_path="./"):
        self.path = base_path + DATA_PATH
        self.logger = logging.getLogger()

    def get_discharge_whole_cycle(self, train_names, test_names, output_capacity=False, scale_test=False, output_time=False):
        train = self._get_data(train_names, output_capacity, output_time)
        test = self._get_data(test_names, output_capacity, output_time)
        train, test = self._scale_x(train, test, scale_test=scale_test)        
        return (train, test)
        
    def _get_data(self, names, output_capacity, output_time=False):
        cycles = []
        for name in names:
            cycle = pd.read_csv(self.path + name + '.csv', skiprows=30)
            cycle.columns = ['Time Stamp','Step','Status','Prog Time','Step Time','Cycle',
                            'Cycle Level','Procedure','Voltage','Current','Temperature','Capacity','WhAccu','Cnt','Empty']
            cycle = cycle[(cycle["Status"] == "TABLE") | (cycle["Status"] == "DCH")]

            max_discharge = abs(min(cycle["Capacity"]))
            cycle["SoC Capacity"] = max_discharge + cycle["Capacity"]
            cycle["SoC Percentage"] = cycle["SoC Capacity"] / max(cycle["SoC Capacity"])
            x = cycle[["Voltage", "Current", "Temperature"]].to_numpy()

            if output_time:
                cycle['Prog Time'] = cycle['Prog Time'].apply(self._time_string_to_seconds)
                cycle['Time in Seconds'] = cycle['Prog Time'] - cycle['Prog Time'][0]

            if output_capacity:
                if output_time:
                    y = cycle[["SoC Capacity", "Time in Seconds"]].to_numpy()
                else:
                    y = cycle[["SoC Capacity"]].to_numpy()
            else:
                if output_time:
                    y = cycle[["SoC Percentage", "Time in Seconds"]].to_numpy()
                else:
                    y = cycle[["SoC Percentage"]].to_numpy()

            if np.isnan(np.min(x)) or np.isnan(np.min(y)):
                self.logger.info("There is a NaN in cycle " + name + ", removing row")
                x = x[~np.isnan(x).any(axis=1)]
                y = y[~np.isnan(y).any(axis=1)].reshape(-1, y.shape[1])

            cycles.append((x, y))

        return cycles

    def _time_string_to_seconds(self, input_string):
        time_parts = input_string.split(':')
        second_parts = time_parts[2].split('.')
        return timedelta(hours=int(time_parts[0]), 
            minutes=int(time_parts[1]), 
            seconds=int(second_parts[0]), 
            microseconds=int(second_parts[1])).total_seconds()

    def _scale_x(self, train, test, scale_test=False):
        for index_feature in range(len(train[0][0][0])):
            feature_min = min([min(cycle[0][:,index_feature]) for cycle in train])
            feature_max = max([max(cycle[0][:,index_feature]) for cycle in train])
            for i in range(len(train)):
                train[i][0][:,index_feature] = (train[i][0][:,index_feature]-feature_min)/(feature_max-feature_min)
            if scale_test:
                for i in range(len(test)):
                    test[i][0][:,index_feature] = (test[i][0][:,index_feature]-feature_min)/(feature_max-feature_min)

        return train, test


    #################################
    #
    # get_stateful_cycle
    #
    #################################
    def get_stateful_cycle(self, cycles, pad_num = 0, steps = 100):
        max_lenght = max(max(len(cycle[0]) for cycle in cycles[0]), max(len(cycle[0]) for cycle in cycles[1]))
        train_x, train_y = self._to_padded_cycle(cycles[0], pad_num, max_lenght)
        test_x, test_y = self._to_padded_cycle(cycles[1], pad_num, max_lenght)
        train_x = self._split_cycle(train_x, steps)
        train_y = self._split_cycle(train_y, steps)
        test_x = self._split_cycle(test_x, steps)
        test_y = self._split_cycle(test_y, steps)
        self.logger.info("Train x: %s, train y: %s | Test x: %s, test y: %s" %
                         (train_x.shape, train_y.shape, test_x.shape, test_y.shape))
        return (train_x, train_y, test_x, test_y)

    def _to_padded_cycle(self, cycles, pad_num, max_lenght):
        x_length = len(cycles[0][0][0])
        y_length = len(cycles[0][1][0])
        x = np.full((len(cycles), max_lenght, x_length), pad_num, dtype=float)
        y = np.full((len(cycles), max_lenght, y_length), pad_num, dtype=float)
        for i, cycle in enumerate(cycles):
            x[i, :cycle[0].shape[0]] = cycle[0]
            y[i, :cycle[1].shape[0]] = cycle[1]
        return x, y

    def _split_cycle(self, cycles, steps):
        features = cycles.shape[2]
        time_steps = cycles.shape[1]
        new_cycles = np.empty((0, time_steps//steps, steps, features), float)
        for cycle in cycles:
            new_cycle = np.empty((0, steps, features), float)
            for i in range(0, len(cycle) - steps, steps):
                next_split = np.array(cycle[i:i + steps]).reshape(1, steps, features)
                new_cycle = np.concatenate((new_cycle, next_split))
            new_cycles = np.concatenate((new_cycles, new_cycle.reshape(1, time_steps//steps, steps, features)))
        return new_cycles


    #################################
    #
    # get_discharge_multiple_step
    #
    #################################
    def get_discharge_multiple_step(self, cycles, steps):
        train_x, train_y = self._split_to_multiple_step(cycles[0], steps)
        test_x, test_y = self._split_to_multiple_step(cycles[1], steps)
        self.logger.info("Train x: %s, train y: %s | Test x: %s, test y: %s" %
                         (train_x.shape, train_y.shape, test_x.shape, test_y.shape))
        return (train_x, train_y, test_x, test_y)

    def _split_to_multiple_step(self, cycles, steps):
        x_length = len(cycles[0][0][0])
        y_length = len(cycles[0][1][0])
        x = np.empty((0, steps, x_length), float)
        y = np.empty((0, steps, y_length), float)
        for cycle in cycles:
            for i in range(0, len(cycle[0]) - steps, steps):
                next_x = np.array(cycle[0][i:i + steps]).reshape(1, steps, x_length)
                next_y = np.array(cycle[1][i:i + steps]).reshape(1, steps, y_length)
                x = np.concatenate((x, next_x))
                y = np.concatenate((y, next_y))
        return x, y

    def keep_only_y_end(self, y, step, is_stateful=False):
        if is_stateful:
            return y[:,:,::step]
        else:
            return y[:,::step]


    

In [5]:
data_path = '/kaggle/input/battery-0-10-25/Battery_0_10_25/'
train_names = [
    '0degC/589_LA92',
    '0degC/589_Mixed1',
    '0degC/589_Mixed2',
    '0degC/589_UDDS',
    '0degC/589_US06',
    '0degC/590_Mixed7',
    '0degC/590_Mixed8',
    
    '10degC/582_LA92',
    '10degC/567_Mixed1',
    '10degC/567_Mixed2',
    '10degC/576_UDDS',
    '10degC/567_US06',
    '10degC/571_Mixed7',
    '10degC/571_Mixed8',
    
    '25degC/551_LA92', 
    '25degC/551_Mixed1', 
    '25degC/551_Mixed2', 
    '25degC/551_UDDS', 
    '25degC/551_US06', 
    '25degC/552_Mixed3',
    '25degC/552_Mixed7', 
    '25degC/552_Mixed8',   
    ]
test_names = [
    '0degC/590_Mixed4',
    '0degC/590_Mixed5',
    '0degC/590_Mixed6',
    
    '10degC/571_Mixed4',
    '10degC/571_Mixed5',
    '10degC/571_Mixed6',

    '25degC/552_Mixed4', 
    '25degC/552_Mixed5', 
    '25degC/552_Mixed6',
    ]

steps = 50

lg_data = LgData(data_path)
cycles = lg_data.get_discharge_whole_cycle(train_names, test_names, output_capacity=False, scale_test=True)
train_x, train_y, test_x, test_y = lg_data.get_discharge_multiple_step(cycles, steps)

train_y = lg_data.keep_only_y_end(train_y, steps)
test_y = lg_data.keep_only_y_end(test_y, steps)

In [6]:
test_x.shape

(11748, 50, 3)

# SELU

In [ ]:
EXPERIMENT = "lstm_huber_50_steps"
ACTIVATION = 'selu'
experiment_name =  EXPERIMENT + '_' + ACTIVATION
print(experiment_name)

os.environ["CUDA_VISIBLE_DEVICES"] = "1"
    
# Model definition
opt = tf.keras.optimizers.Adam(lr=0.00001)

model = Sequential()
model.add(LSTM(256, activation=ACTIVATION,
                return_sequences=True,
                input_shape=(train_x.shape[1], train_x.shape[2])))
model.add(LSTM(256, activation=ACTIVATION, return_sequences=False))
model.add(Dense(256, activation=ACTIVATION))
model.add(Dense(128, activation=ACTIVATION))
model.add(Dense(1, activation=ACTIVATION))
model.summary()

model.compile(optimizer=opt, loss='huber', metrics=['mse', 'mae', 'mape', tf.keras.metrics.RootMeanSquaredError(name='rmse')])

es = EarlyStopping(monitor='val_loss', patience=30)
model_path = '/kaggle/working/results/trained_model/%s_best.h5' % experiment_name
mc = ModelCheckpoint(model_path, 
                             save_best_only=True, 
                             monitor='val_loss')

In [ ]:
history = model.fit(train_x, train_y, 
                                epochs=1000, 
                                batch_size=500, 
                                verbose=2,
                                validation_split=0.2,
                                callbacks = [es, mc]
                               )

In [10]:
#model.save('/kaggle/working/trained_model/%s.h5' % experiment_name)

hist_df = pd.DataFrame(history.history)
hist_csv_file = '/kaggle/working/results/trained_model/%s_history.csv' % experiment_name
with open(hist_csv_file, mode='w') as f:
    hist_df.to_csv(f)
    print("Completed")

Completed


In [ ]:
model = tf.keras.models.load_model('/kaggle/working/results/trained_model/lstm_huber_50_steps_selu_best.h5')
predictions = model.predict(test_x)

# Calculate and print metrics
calculate_and_print_metrics(test_y.reshape(-1,1), predictions)

# Plot true vs predicted values
plot_true_vs_predicted(test_y.reshape(-1,1), predictions, "SELU")

# Sigmoid

In [13]:
EXPERIMENT = "lstm_huber_50_steps"
ACTIVATION = 'sigmoid'
experiment_name =  EXPERIMENT + '_' + ACTIVATION
print(experiment_name)

os.environ["CUDA_VISIBLE_DEVICES"] = "1"
    
# Model definition
opt = tf.keras.optimizers.Adam(lr=0.00001)

model = Sequential()
model.add(LSTM(256, activation=ACTIVATION,
                return_sequences=True,
                input_shape=(train_x.shape[1], train_x.shape[2])))
model.add(LSTM(256, activation=ACTIVATION, return_sequences=False))
model.add(Dense(256, activation=ACTIVATION))
model.add(Dense(128, activation=ACTIVATION))
model.add(Dense(1, activation=ACTIVATION))
model.summary()

model.compile(optimizer=opt, loss='huber', metrics=['mse', 'mae', 'mape', tf.keras.metrics.RootMeanSquaredError(name='rmse')])

es = EarlyStopping(monitor='val_loss', patience=30)
model_path = '/kaggle/working/results/trained_model/%s_best.h5' % experiment_name
mc = ModelCheckpoint(model_path, 
                             save_best_only=True, 
                             monitor='val_loss')

lstm_huber_50_steps_sigmoid
Model: "sequential_1"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 lstm_2 (LSTM)               (None, 50, 256)           266240    
                                                                 
 lstm_3 (LSTM)               (None, 256)               525312    
                                                                 
 dense_3 (Dense)             (None, 256)               65792     
                                                                 
 dense_4 (Dense)             (None, 128)               32896     
                                                                 
 dense_5 (Dense)             (None, 1)                 129       
                                                                 
Total params: 890369 (3.40 MB)
Trainable params: 890369 (3.40 MB)
Non-trainable params: 0 (0.00 Byte)
________________________________________________________

In [ ]:
history = model.fit(train_x, train_y, 
                                epochs=1000, 
                                batch_size=500, 
                                verbose=2,
                                validation_split=0.2,
                                callbacks = [es, mc]
                               )

In [15]:
hist_df = pd.DataFrame(history.history)
hist_csv_file = '/kaggle/working/results/trained_model/%s_history.csv' % experiment_name
with open(hist_csv_file, mode='w') as f:
    hist_df.to_csv(f)
    print("Completed")

Completed


In [ ]:
model = tf.keras.models.load_model('/kaggle/working/results/trained_model/lstm_huber_50_steps_sigmoid_best.h5')
predictions = model.predict(test_x)

# Calculate and print metrics
calculate_and_print_metrics(test_y.reshape(-1,1), predictions)

# Plot true vs predicted values
plot_true_vs_predicted(test_y.reshape(-1,1), predictions, "Sigmoid")

# Tanh

In [7]:
EXPERIMENT = "lstm_huber_50_steps"
ACTIVATION = 'tanh'
experiment_name =  EXPERIMENT + '_' + ACTIVATION
print(experiment_name)

os.environ["CUDA_VISIBLE_DEVICES"] = "1"
    
# Model definition
opt = tf.keras.optimizers.Adam(lr=0.00001)

model = Sequential()
model.add(LSTM(256, activation=ACTIVATION,
                return_sequences=True,
                input_shape=(train_x.shape[1], train_x.shape[2])))
model.add(LSTM(256, activation=ACTIVATION, return_sequences=False))
model.add(Dense(256, activation=ACTIVATION))
model.add(Dense(128, activation=ACTIVATION))
model.add(Dense(1, activation=ACTIVATION))
model.summary()

model.compile(optimizer=opt, loss='huber', metrics=['mse', 'mae', 'mape', tf.keras.metrics.RootMeanSquaredError(name='rmse')])

es = EarlyStopping(monitor='val_loss', patience=30)
model_path = '/kaggle/working/results/trained_model/%s_best.h5' % experiment_name
mc = ModelCheckpoint(model_path, 
                             save_best_only=True, 
                             monitor='val_loss')

lstm_huber_50_steps_tanh
Model: "sequential"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 lstm (LSTM)                 (None, 50, 256)           266240    
                                                                 
 lstm_1 (LSTM)               (None, 256)               525312    
                                                                 
 dense (Dense)               (None, 256)               65792     
                                                                 
 dense_1 (Dense)             (None, 128)               32896     
                                                                 
 dense_2 (Dense)             (None, 1)                 129       
                                                                 
Total params: 890369 (3.40 MB)
Trainable params: 890369 (3.40 MB)
Non-trainable params: 0 (0.00 Byte)
_____________________________________________________________

In [ ]:
history = model.fit(train_x, train_y, 
                                epochs=1000, 
                                batch_size=500, 
                                verbose=2,
                                validation_split=0.2,
                                callbacks = [es, mc]
                               )

In [9]:
hist_df = pd.DataFrame(history.history)
hist_csv_file = '/kaggle/working/results/trained_model/%s_history.csv' % experiment_name
with open(hist_csv_file, mode='w') as f:
    hist_df.to_csv(f)
    print("Completed")

Completed


In [ ]:
model = tf.keras.models.load_model('/kaggle/working/results/trained_model/lstm_huber_50_steps_tanh_best.h5')
predictions = model.predict(test_x)

# Calculate and print metrics
calculate_and_print_metrics(test_y.reshape(-1,1), predictions)

# Plot true vs predicted values
plot_true_vs_predicted(test_y.reshape(-1,1), predictions, "Tanh")

# Mish

In [12]:
EXPERIMENT = "lstm_huber_50_steps"
ACTIVATION = 'mish'
experiment_name =  EXPERIMENT + '_' + ACTIVATION
print(experiment_name)

os.environ["CUDA_VISIBLE_DEVICES"] = "1"
    
# Model definition
opt = tf.keras.optimizers.Adam(lr=0.00001)

model = Sequential()
model.add(LSTM(256, activation=ACTIVATION,
                return_sequences=True,
                input_shape=(train_x.shape[1], train_x.shape[2])))
model.add(LSTM(256, activation=ACTIVATION, return_sequences=False))
model.add(Dense(256, activation=ACTIVATION))
model.add(Dense(128, activation=ACTIVATION))
model.add(Dense(1, activation=ACTIVATION))
model.summary()

model.compile(optimizer=opt, loss='huber', metrics=['mse', 'mae', 'mape', tf.keras.metrics.RootMeanSquaredError(name='rmse')])

es = EarlyStopping(monitor='val_loss', patience=30)
model_path = '/kaggle/working/results/trained_model/%s_best.h5' % experiment_name
mc = ModelCheckpoint(model_path, 
                             save_best_only=True, 
                             monitor='val_loss')

lstm_huber_50_steps_mish
Model: "sequential_2"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 lstm_4 (LSTM)               (None, 50, 256)           266240    
                                                                 
 lstm_5 (LSTM)               (None, 256)               525312    
                                                                 
 dense_6 (Dense)             (None, 256)               65792     
                                                                 
 dense_7 (Dense)             (None, 128)               32896     
                                                                 
 dense_8 (Dense)             (None, 1)                 129       
                                                                 
Total params: 890369 (3.40 MB)
Trainable params: 890369 (3.40 MB)
Non-trainable params: 0 (0.00 Byte)
___________________________________________________________

In [ ]:
history = model.fit(train_x, train_y, 
                                epochs=1000, 
                                batch_size=500, 
                                verbose=2,
                                validation_split=0.2,
                                callbacks = [es, mc]
                               )

In [14]:
hist_df = pd.DataFrame(history.history)
hist_csv_file = '/kaggle/working/results/trained_model/%s_history.csv' % experiment_name
with open(hist_csv_file, mode='w') as f:
    hist_df.to_csv(f)
    print("Completed")

Completed


In [ ]:
model = tf.keras.models.load_model("/kaggle/working/results/trained_model/lstm_huber_50_steps_mish_best.h5")
predictions = model.predict(test_x)

# Calculate and print metrics
calculate_and_print_metrics(test_y.reshape(-1,1), predictions)

# Plot true vs predicted values
plot_true_vs_predicted(test_y.reshape(-1,1), predictions, 'mish')

# Swish

In [16]:
EXPERIMENT = "lstm_huber_50_steps"
ACTIVATION = 'swish'
experiment_name =  EXPERIMENT + '_' + ACTIVATION
print(experiment_name)

os.environ["CUDA_VISIBLE_DEVICES"] = "1"
    
# Model definition
opt = tf.keras.optimizers.Adam(lr=0.00001)

model = Sequential()
model.add(LSTM(256, activation=ACTIVATION,
                return_sequences=True,
                input_shape=(train_x.shape[1], train_x.shape[2])))
model.add(LSTM(256, activation=ACTIVATION, return_sequences=False))
model.add(Dense(256, activation=ACTIVATION))
model.add(Dense(128, activation=ACTIVATION))
model.add(Dense(1, activation=ACTIVATION))
model.summary()

model.compile(optimizer=opt, loss='huber', metrics=['mse', 'mae', 'mape', tf.keras.metrics.RootMeanSquaredError(name='rmse')])

es = EarlyStopping(monitor='val_loss', patience=30)
model_path = '/kaggle/working/results/trained_model/%s_best.h5' % experiment_name
mc = ModelCheckpoint(model_path, 
                             save_best_only=True, 
                             monitor='val_loss')

lstm_huber_50_steps_swish
Model: "sequential_3"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 lstm_6 (LSTM)               (None, 50, 256)           266240    
                                                                 
 lstm_7 (LSTM)               (None, 256)               525312    
                                                                 
 dense_9 (Dense)             (None, 256)               65792     
                                                                 
 dense_10 (Dense)            (None, 128)               32896     
                                                                 
 dense_11 (Dense)            (None, 1)                 129       
                                                                 
Total params: 890369 (3.40 MB)
Trainable params: 890369 (3.40 MB)
Non-trainable params: 0 (0.00 Byte)
__________________________________________________________

In [ ]:
history = model.fit(train_x, train_y, 
                                epochs=1000, 
                                batch_size=500, 
                                verbose=2,
                                validation_split=0.2,
                                callbacks = [es, mc]
                               )

In [18]:
hist_df = pd.DataFrame(history.history)
hist_csv_file = '/kaggle/working/results/trained_model/%s_history.csv' % experiment_name
with open(hist_csv_file, mode='w') as f:
    hist_df.to_csv(f)
    print("Completed")

Completed


In [ ]:
model = tf.keras.models.load_model("/kaggle/working/results/trained_model/lstm_huber_50_steps_swish_best.h5")
predictions = model.predict(test_x)

# Calculate and print metrics
calculate_and_print_metrics(test_y.reshape(-1,1), predictions)

# Plot true vs predicted values
plot_true_vs_predicted(test_y.reshape(-1,1), predictions, "Swish")

# RELU

In [20]:
EXPERIMENT = "lstm_huber_50_steps"
ACTIVATION = 'relu'
experiment_name =  EXPERIMENT + '_' + ACTIVATION
print(experiment_name)

os.environ["CUDA_VISIBLE_DEVICES"] = "1"
    
# Model definition
opt = tf.keras.optimizers.Adam(lr=0.00001)

model = Sequential()
model.add(LSTM(256, activation=ACTIVATION,
                return_sequences=True,
                input_shape=(train_x.shape[1], train_x.shape[2])))
model.add(LSTM(256, activation=ACTIVATION, return_sequences=False))
model.add(Dense(256, activation=ACTIVATION))
model.add(Dense(128, activation=ACTIVATION))
model.add(Dense(1, activation=ACTIVATION))
model.summary()

model.compile(optimizer=opt, loss='huber', metrics=['mse', 'mae', 'mape', tf.keras.metrics.RootMeanSquaredError(name='rmse')])

es = EarlyStopping(monitor='val_loss', patience=30)
model_path = '/kaggle/working/results/trained_model/%s_best.h5' % experiment_name
mc = ModelCheckpoint(model_path, 
                             save_best_only=True, 
                             monitor='val_loss')

lstm_huber_50_steps_relu
Model: "sequential_4"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 lstm_8 (LSTM)               (None, 50, 256)           266240    
                                                                 
 lstm_9 (LSTM)               (None, 256)               525312    
                                                                 
 dense_12 (Dense)            (None, 256)               65792     
                                                                 
 dense_13 (Dense)            (None, 128)               32896     
                                                                 
 dense_14 (Dense)            (None, 1)                 129       
                                                                 
Total params: 890369 (3.40 MB)
Trainable params: 890369 (3.40 MB)
Non-trainable params: 0 (0.00 Byte)
___________________________________________________________

In [ ]:
history = model.fit(train_x, train_y, 
                                epochs=1000, 
                                batch_size=500, 
                                verbose=2,
                                validation_split=0.2,
                                callbacks = [es, mc]
                               )

In [22]:
hist_df = pd.DataFrame(history.history)
hist_csv_file = '/kaggle/working/results/trained_model/%s_history.csv' % experiment_name
with open(hist_csv_file, mode='w') as f:
    hist_df.to_csv(f)
    print("Completed")

Completed


In [ ]:
model = tf.keras.models.load_model("/kaggle/working/results/trained_model/lstm_huber_50_steps_relu_best.h5")
predictions = model.predict(test_x)

# Calculate and print metrics
calculate_and_print_metrics(test_y.reshape(-1,1), predictions)

# Plot true vs predicted values
plot_true_vs_predicted(test_y.reshape(-1,1), predictions, "ReLU")

# Leaky Relu

In [24]:
EXPERIMENT = "lstm_huber_50_steps"
ACTIVATION = 'leaky_relu'
experiment_name =  EXPERIMENT + '_' + ACTIVATION
print(experiment_name)

os.environ["CUDA_VISIBLE_DEVICES"] = "1"
    
# Model definition
opt = tf.keras.optimizers.Adam(lr=0.00001)

model = Sequential()
model.add(LSTM(256, activation=ACTIVATION,
                return_sequences=True,
                input_shape=(train_x.shape[1], train_x.shape[2])))
model.add(LSTM(256, activation=ACTIVATION, return_sequences=False))
model.add(Dense(256, activation=ACTIVATION))
model.add(Dense(128, activation=ACTIVATION))
model.add(Dense(1, activation=ACTIVATION))
model.summary()

model.compile(optimizer=opt, loss='huber', metrics=['mse', 'mae', 'mape', tf.keras.metrics.RootMeanSquaredError(name='rmse')])

es = EarlyStopping(monitor='val_loss', patience=30)
model_path = '/kaggle/working/results/trained_model/%s_best.h5' % experiment_name
mc = ModelCheckpoint(model_path, 
                             save_best_only=True, 
                             monitor='val_loss')

lstm_huber_50_steps_leaky_relu
Model: "sequential_5"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 lstm_10 (LSTM)              (None, 50, 256)           266240    
                                                                 
 lstm_11 (LSTM)              (None, 256)               525312    
                                                                 
 dense_15 (Dense)            (None, 256)               65792     
                                                                 
 dense_16 (Dense)            (None, 128)               32896     
                                                                 
 dense_17 (Dense)            (None, 1)                 129       
                                                                 
Total params: 890369 (3.40 MB)
Trainable params: 890369 (3.40 MB)
Non-trainable params: 0 (0.00 Byte)
_____________________________________________________

In [ ]:
history = model.fit(train_x, train_y, 
                                epochs=1000, 
                                batch_size=500, 
                                verbose=2,
                                validation_split=0.2,
                                callbacks = [es, mc]
                               )

In [26]:
hist_df = pd.DataFrame(history.history)
hist_csv_file = '/kaggle/working/results/trained_model/%s_history.csv' % experiment_name
with open(hist_csv_file, mode='w') as f:
    hist_df.to_csv(f)
    print("Completed")

Completed


In [ ]:
model = tf.keras.models.load_model("/kaggle/working/results/trained_model/lstm_huber_50_steps_leaky_relu_best.h5")
predictions = model.predict(test_x)

# Calculate and print metrics
calculate_and_print_metrics(test_y.reshape(-1,1), predictions)

# Plot true vs predicted values
plot_true_vs_predicted(test_y.reshape(-1,1), predictions, "LeakyReLU")

# ELU

In [8]:
EXPERIMENT = "lstm_huber_50_steps"
ACTIVATION = 'elu'
experiment_name =  EXPERIMENT + '_' + ACTIVATION
print(experiment_name)

os.environ["CUDA_VISIBLE_DEVICES"] = "1"
    
# Model definition
opt = tf.keras.optimizers.Adam(lr=0.00001)

model = Sequential()
model.add(LSTM(256, activation=ACTIVATION,
                return_sequences=True,
                input_shape=(train_x.shape[1], train_x.shape[2])))
model.add(LSTM(256, activation=ACTIVATION, return_sequences=False))
model.add(Dense(256, activation=ACTIVATION))
model.add(Dense(128, activation=ACTIVATION))
model.add(Dense(1, activation=ACTIVATION))
model.summary()

model.compile(optimizer=opt, loss='huber', metrics=['mse', 'mae', 'mape', tf.keras.metrics.RootMeanSquaredError(name='rmse')])

es = EarlyStopping(monitor='val_loss', patience=30)
model_path = '/kaggle/working/results/trained_model/%s_best.h5' % experiment_name
mc = ModelCheckpoint(model_path, 
                             save_best_only=True, 
                             monitor='val_loss')

lstm_huber_50_steps_elu
Model: "sequential"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 lstm (LSTM)                 (None, 50, 256)           266240    
                                                                 
 lstm_1 (LSTM)               (None, 256)               525312    
                                                                 
 dense (Dense)               (None, 256)               65792     
                                                                 
 dense_1 (Dense)             (None, 128)               32896     
                                                                 
 dense_2 (Dense)             (None, 1)                 129       
                                                                 
Total params: 890369 (3.40 MB)
Trainable params: 890369 (3.40 MB)
Non-trainable params: 0 (0.00 Byte)
______________________________________________________________

In [ ]:
history = model.fit(train_x, train_y, 
                                epochs=1000, 
                                batch_size=500, 
                                verbose=2,
                                validation_split=0.2,
                                callbacks = [es, mc]
                               )

In [10]:
hist_df = pd.DataFrame(history.history)
hist_csv_file = '/kaggle/working/results/trained_model/%s_history.csv' % experiment_name
with open(hist_csv_file, mode='w') as f:
    hist_df.to_csv(f)
    print("Completed")

Completed


In [ ]:
model = tf.keras.models.load_model("/kaggle/working/results/trained_model/lstm_huber_50_steps_elu_best.h5")
predictions = model.predict(test_x)

# Calculate and print metrics
calculate_and_print_metrics(test_y.reshape(-1,1), predictions)

# Plot true vs predicted values
plot_true_vs_predicted(test_y.reshape(-1,1), predictions, 'Elu')